# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [3]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [8]:
# 1
print("Строка и столбцов;" , df.shape)

Строка и столбцов; (5463, 8)


In [9]:
df.info

<bound method DataFrame.info of       order_id        order_date customer_id       city     category   amount  \
0       101674  2026-09-01 08:00      C00681     Алматы       Одежда   8070.0   
1       101214  2026-09-01 08:00      C01210     Астана      Красота   3630.0   
2       102424  2026-09-01 08:00      C01408  Караганда          Дом   9120.0   
3       102333  2026-09-01 08:00      C01529     Алматы       Одежда   9350.0   
4       102564  2026-09-01 09:00      C00657     Астана          Дом  11010.0   
...        ...               ...         ...        ...          ...      ...   
5458    104168  2026-10-31 21:00      C01057     Алматы       Одежда  11430.0   
5459    104648  2026-10-31 21:00      C00629  Караганда        Книги   3640.0   
5460    104615  2026-10-31 22:00      C00557     Астана  Электроника  22150.0   
5461    103460  2026-10-31 22:00      C02162     Алматы  Электроника  24670.0   
5462    103501  2026-10-31 22:00      C00269     Алматы  Электроника  14910.0

In [11]:
df['delivery_days'].isna().sum()

np.int64(160)

In [12]:
df[['city', 'category', 'status', 'customer_id']].nunique()

city             14
category          5
status            2
customer_id    2003
dtype: int64

Тут проблема разные написание города например:Алматы алматы АЛМАТЫ итд 
и еще есть правильные типы например amount не числовой тип. 
у нас есть пропуски в нашем датасете 

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [13]:
# 2
paid = df[
    (df['status'] == 'оплачен') &
    (df['customer_id'] != 'TEST')
].copy()

In [16]:
paid['amount'] = (
    paid['amount'].replace(' ', '', regex=False)
    .astype(float)
)

In [17]:
paid['amount'].mean()

np.float64(14672.391348704346)

In [18]:
paid['amount'].median()

10300.0

In [22]:
print("q1:", paid['amount'].quantile(0.25))
print("q2:", paid['amount'].quantile(0.75))


q1: 5880.0
q2: 18690.0


Мы выбрали медиану, потому что она менее чувствительна к выбросам и лучше отражает типичную стоимость заказа в нашем датасете. Среднее значение мы также рассчитали, чтобы дополнительно показать общую картину расходов.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [23]:
# 3
df['status'].value_counts()

status
оплачен    4904
отменён     559
Name: count, dtype: int64

In [24]:
cancel_rate = (df['status'] == 'отменён').mean() * 100

print("Доля отмен:", cancel_rate)

Доля отмен: 10.232473000183049


In [25]:
df['city'].value_counts()

city
Алматы       1927
Астана       1226
Шымкент       711
Караганда     538
Актобе        397
Атырау        146
астана         84
Astana         80
Almaty         69
 Алматы        66
алматы         64
АЛМАТЫ         62
шымкент        54
Shymkent       39
Name: count, dtype: int64

In [29]:
shymkent_names = ['Шымкент', 'шымкент', 'Shymkent']
shymkent = df[df['city'].isin(shymkent_names)]
almaty_names = ['Алматы', 'алматы', 'АЛМАТЫ', 'Almaty', ' Алматы']
almaty = df[df['city'].isin(almaty_names)]

In [30]:
shymkent_cancel = (shymkent['status'] == 'отменён').mean() * 100
print("Отмены в Шымкенте:", shymkent_cancel)
almaty_cancel = (almaty['status'] == 'отменён').mean() * 100
print("Отмены в Алматы:", almaty_cancel)

Отмены в Шымкенте: 20.398009950248756
Отмены в Алматы: 7.861060329067642


In [32]:
ratio = shymkent_cancel / almaty_cancel
print("Во сколько раз:", ratio)

Во сколько раз: 2.5948166146014113


В Шымкенте самый высокий показатель отмены заказа. Этот показатель примерно в 2,6 раза выше, чем в Алматы. Это указывает на потенциальную проблему с отменой заказа в Шымкенте, которая требует дальнейшего расследования

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [34]:
# 4
paid['order_date'] = pd.to_datetime(paid['order_date'])
paid['month'] = paid['order_date'].dt.month

In [35]:
paid['city'] = paid['city'].str.strip().str.lower()
paid['city'] = paid['city'].replace({
    'almaty': 'алматы',
    'shymkent': 'шымкент'
})

In [37]:
cities = ['алматы', 'шымкент']
paid = paid[paid['city'].isin(cities)]
paid = paid[paid['month'].isin([9, 10])]

In [38]:
paid['amount'] = (
    paid['amount'].replace(' ', '', regex=False)
    .astype(float)
)

In [39]:
revenue = paid.groupby(
    ['city', 'month']
)['amount'].sum()

print(revenue)

city     month
алматы   9        15523940.0
         10       14643020.0
шымкент  9         4949800.0
         10        4044010.0
Name: amount, dtype: float64


In [41]:
result = paid.groupby(['city', 'month'])['amount'].sum().unstack()
result['change_%'] = (
    (result[10] - result[9]) / result[9]
) * 100
print(result)

month             9          10   change_%
city                                      
алматы   15523940.0  14643020.0  -5.674590
шымкент   4949800.0   4044010.0 -18.299527


In [43]:
ratio = abs(result.loc['шымкент', 'change_%']) / abs(result.loc['алматы', 'change_%'])
print(ratio)

3.224819088153989


Выручка снизилась в обоих городах. В Шымкенте наблюдалось более сильное снижение. Его процентное снижение было примерно в 3,22 раза больше, чем в Алматы.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [44]:
# 5
(df["city"] == "Шымкент") & (df["status"] == "отменён")

0       False
1       False
2       False
3       False
4       False
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Length: 5463, dtype: bool

In [45]:
df['city_clean'] = (
    df['city']
    .str.strip()
    .str.lower()
    .replace({'shymkent': 'шымкент'})
)

In [46]:
cancel_sh = df[
    (df['city_clean'] == 'шымкент') &
    (df['status'] == 'отменён')
]

print(len(cancel_sh))

164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [49]:
# 6
df['customer_id'].value_counts().head(1)

customer_id
C01818    11
Name: count, dtype: int64

In [50]:
customer_orders = df.loc[
    df['customer_id'] == 'C01818',
    ['order_id', 'order_date', 'city', 'amount', 'status']
]
print(customer_orders)

      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен


In [51]:
customer_orders[
    customer_orders['order_id'].duplicated(keep=False)
]

,order_id,order_date,city,amount,status
1371,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,101504,2026-09-15 21:00,Алматы,8890.0,оплачен


Повторяющиеся заказы могут искусственно увеличить общую выручку, поскольку один и тот же заказ может быть учтен более одного раза. В этом случае выручка для клиента C01818 будет завышена на 8 890, если повторный заказ будет учтен дважды.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [53]:
# 7
paid['amount'] = pd.to_numeric(paid['amount'])
Q1 = paid['amount'].quantile(0.25)
Q3 = paid['amount'].quantile(0.75)

IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

Q1: 5900.0
Q3: 18800.0
IQR: 12900.0


In [54]:
upper_bound = Q3 + 1.5 * IQR
print("Верхняя граница:", upper_bound)
outliers = paid[paid['amount'] > upper_bound]
print("Количество выбросов:", len(outliers))

Верхняя граница: 38150.0
Количество выбросов: 162


In [56]:
percentage = len(outliers) / len(paid) * 100
print("Доля выбросов:", percentage, "%")

Доля выбросов: 6.106294760648322 %


In [57]:
outliers['category'].value_counts()

category
Электроника    156
Одежда           5
Дом              1
Name: count, dtype: int64

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8